# 06 – Baseline: Wie gut wäre eine ganz einfache Schätzung?

**Ziel.** Zwei sehr einfache Schätzungen der Zahl sichtbarer Parkfelder je Ausschnitt aufstellen, **bevor** ein Netz gelernt hat:

| Name | Schätzung |
| --- | --- |
| **B0** | überall 0 Felder |
| **B1** | in jedem Ausschnitt gleich viele Felder: der Durchschnitt der Lern-Ausschnitte |

**Warum.** Ein Fehler von "2 Feldern je Ausschnitt" sagt allein wenig. Erst der Vergleich mit einer einfachen Schätzung zeigt, ob das Netz aus dem Bild etwas gelernt hat. Ein Netz, das B1 nicht schlägt, braucht das Bild gar nicht (`01b`, 7.9).

**Eingaben:** `kaestchen.gpkg` (Notebook 04), `aufteilung.csv` (Notebook 02).
**Ausgaben:** `ergebnisse/baseline.csv` (B1 je Versuch und der Fehler beider Schätzungen auf der Validierung).

In [1]:
from pathlib import Path           # Pfade zusammensetzen

ROOT = Path.cwd().parent                             # Projektordner (eine Ebene über notebooks/)
VERARB = ROOT / "daten" / "verarbeitet"              # abgeleitete Daten
ERG = ROOT / "ergebnisse"                            # Ergebnisse

import pandas as pd                # Tabellen
import geopandas as gpd            # Tabellen mit Geometrie

SPALTE = {"A": "kat_2021", "B": "kat_2024_25"}       # Spalte der Kategorie je Versuch (A: Bild 2021, B: Bild 2024/25)

## 1. Sichtbare Felder je Ausschnitt

**Schritt 1: zählen.** Für jeden Ausschnitt zählen wir die Felder der Kategorie `sichtbar`, deren Ring im Ausschnitt liegt. Ausschnitte ohne Feld bekommen eine 0 (`reindex` ergänzt sie, `fillna(0)` setzt die Null).

Vorher: Felder → nachher: eine Zeile je Ausschnitt mit der Zahl sichtbarer Felder je Versuch.

In [2]:
felder = gpd.read_file(VERARB / "kaestchen.gpkg", ignore_geometry=True)     # Felder mit Kategorie (ohne Formen)
ausschnitte = pd.read_csv(VERARB / "aufteilung.csv").set_index("kachel")   # Gruppe je Ausschnitt
for versuch, spalte in SPALTE.items():
    sichtbar = felder[felder[spalte] == "sichtbar"].kachel.value_counts()  # sichtbare Felder je Ausschnitt
    ausschnitte[f"felder_{versuch}"] = sichtbar.reindex(ausschnitte.index).fillna(0).astype(int)
print(f"Vorher: {len(felder)} Felder → nachher: {len(ausschnitte)} Ausschnitte")
ausschnitte.groupby("gruppe")[["felder_A", "felder_B"]].agg(["sum", "mean"]).round(2)

Vorher: 3269 Felder → nachher: 691 Ausschnitte


felder_A       felder_B      
                sum  mean      sum  mean
gruppe                                  
lernen         1655  3.54     1091  2.34
testen          708  5.17      500  3.65
validieren      345  3.97      242  2.78

## 2. Die beiden Schätzungen

**Schritt 1: B1 aus den Lern-Ausschnitten.** B1 ist der Durchschnitt der sichtbaren Felder je Lern-Ausschnitt. Die Validier- und Testausschnitte werden dafür nicht angesehen.

**Schritt 2: der Fehler auf der Validierung.** Für jeden Validier-Ausschnitt: Schätzung minus Wahrheit. Zwei Kennzahlen (`01b`, 7.9):
- **MAE** (mittlerer Betrag des Fehlers): um wie viele Felder liegt die Schätzung je Ausschnitt im Mittel daneben?
- **Bias**: Summe der Schätzungen ÷ Summe der wahren Felder − 1. Minus heisst zu wenig, plus zu viel.

Die Testausschnitte bleiben unberührt; dort werden die Baselines erst in Notebook 10 gerechnet, zusammen mit dem Netz.

In [3]:
lernen = ausschnitte[ausschnitte.gruppe == "lernen"]
validieren = ausschnitte[ausschnitte.gruppe == "validieren"]
zeilen = []
for versuch in SPALTE:
    b1 = lernen[f"felder_{versuch}"].mean()                          # Durchschnitt je Lern-Ausschnitt
    wahr = validieren[f"felder_{versuch}"]                           # Wahrheit je Validier-Ausschnitt
    zeilen.append({"versuch": versuch, "b1_je_ausschnitt": round(b1, 3),
                   "mae_b0_validierung": round(wahr.abs().mean(), 3),              # B0 schätzt 0: Fehler = Wahrheit
                   "mae_b1_validierung": round((b1 - wahr).abs().mean(), 3),
                   "bias_b1_validierung": round(b1 * len(wahr) / wahr.sum() - 1, 3)})
baseline = pd.DataFrame(zeilen).set_index("versuch")
baseline.to_csv(ERG / "baseline.csv")
baseline

,b1_je_ausschnitt,mae_b0_validierung,mae_b1_validierung,bias_b1_validierung
versuch,,,,
A,3.544,3.966,4.501,-0.106
B,2.336,2.782,3.400,-0.160


**Wie man die Tabelle liest.** `b1_je_ausschnitt` ist die Zahl, die B1 für jeden Ausschnitt schätzt. Die MAE von B0 ist einfach die mittlere Zahl der Felder je Validier-Ausschnitt. Die MAE von B1 ist **nicht kleiner** als die von B0: Die Felder sind sehr ungleich verteilt (viele Ausschnitte ohne Feld, wenige mit vielen), ein Durchschnitt trifft darum fast keinen Ausschnitt. Beide Baselines bleiben stehen, wie sie sind.

## Was wir gesehen haben

- B1 je Versuch und die Fehler beider Schätzungen auf der Validierung stehen in der Tabelle und in `ergebnisse/baseline.csv`.
- Das ist die Messlatte: Das Netz muss in den Testblöcken eine deutlich kleinere MAE erreichen als B0 und B1, sonst hat es aus dem Bild nichts Brauchbares gelernt.

## Wie es weitergeht

Notebook 07 lässt das Netz lernen: viermal, für zwei Bilder und zwei Auflösungen.